In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [2]:
def generate_random_bits(num_bits):
    simulator = BasicSimulator()
    random_bits = []
    for _ in range(num_bits):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)

        compiled_circuit = transpile(qc, simulator);
        job = simulator.run(compiled_circuit, shots=1)
        result = job.result()
        counts = result.get_counts(qc)

        if '0' in counts:
            random_bits.append(0)
        else:
            random_bits.append(1)
    return random_bits

In [3]:
def alice_prepares_qubits(message_length):
    message_bits = generate_random_bits(message_length)
    alice_bases = generate_random_bits(message_length)

    prepared_qubits = []
    for i in range(message_length):
        qc = QuantumCircuit(1, 1)
        if message_bits[i] == 1:
            qc.x(0)
        if alice_bases[i] == 1:
            qc.h(0)
        prepared_qubits.append(qc)

    return message_bits, alice_bases, prepared_qubits

In [6]:
def bob_measures_qubits(prepared_qubits):
    num_qubits = len(prepared_qubits)
    bob_bases = generate_random_bits(num_qubits)

    bob_measurement_results = []
    simulator = BasicSimulator()

    for i in range(num_qubits):
        qc = prepared_qubits[i]

        if bob_bases[i] == 1:
            qc.h(0)

        qc.measure(0, 0);

        compiled_circuit = transpile(qc, simulator)
        job = simulator.run(compiled_circuit, shots=1)
        result = job.result()
        counts = result.get_counts(qc)

        if '0' in counts:
            bob_measurement_results.append(0)
        else:
            bob_measurement_results.append(1)

    return bob_bases, bob_measurement_results

In [5]:
def reconcile_keys(alice_bases, bob_bases, bob_measurement_results):
    raw_key = []
    matching_indices = []
    for i in range(len(alice_bases)):
        if alice_bases[i] == bob_bases[i]:
            raw_key.append(bob_measurement_results[i])
            matching_indices.append(i)

    return raw_key, matching_indices